In [1]:
import numpy as np
import pandas as pd
import ast
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import pickle

ps = PorterStemmer()

def stem(text):
    return " ".join(ps.stem(i) for i in text.split())



print("Loading TMDB data...")
tmdb_movies = pd.read_csv('tmdb_5000_movies.csv')
tmdb_credits = pd.read_csv('tmdb_5000_credits.csv')

tmdb = tmdb_movies.merge(tmdb_credits, on='title')
tmdb = tmdb[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
tmdb.dropna(inplace=True)

def convert(obj):
    return [i['name'] for i in ast.literal_eval(obj)]

def convert3(obj):
    return [i['name'] for i in ast.literal_eval(obj)[:3]]

def fetch_director(obj):
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            return [i['name']]
    return []

tmdb['genres']   = tmdb['genres'].apply(convert)
tmdb['keywords'] = tmdb['keywords'].apply(convert)
tmdb['cast']     = tmdb['cast'].apply(convert3)
tmdb['crew']     = tmdb['crew'].apply(fetch_director)
tmdb['overview'] = tmdb['overview'].apply(lambda x: x.split())


tmdb['genres_stored'] = tmdb['genres'].apply(lambda x: [i.lower() for i in x])


tmdb['genres']   = tmdb['genres'].apply(lambda x: [i.replace(" ", "").lower() for i in x])
tmdb['keywords'] = tmdb['keywords'].apply(lambda x: [i.replace(" ", "").lower() for i in x])
tmdb['cast']     = tmdb['cast'].apply(lambda x: [i.replace(" ", "").lower() for i in x])
tmdb['crew']     = tmdb['crew'].apply(lambda x: [i.replace(" ", "").lower() for i in x])

tmdb['tags'] = tmdb['overview'] + tmdb['genres'] + tmdb['keywords'] + tmdb['cast'] + tmdb['crew']

tmdb_df = tmdb[['movie_id', 'title', 'tags', 'genres_stored']].copy()
tmdb_df.rename(columns={'genres_stored': 'genres'}, inplace=True)

tmdb_df['tags']        = tmdb_df['tags'].apply(lambda x: " ".join(x).lower())
tmdb_df['tags']        = tmdb_df['tags'].apply(stem)
tmdb_df['title_clean'] = tmdb_df['title'].astype(str).str.lower().str.strip()
tmdb_df = tmdb_df.reset_index(drop=True)

print("Building TMDB TF-IDF + kNN model...")
tfidf_tmdb = TfidfVectorizer(max_features=5000, stop_words='english')
X_tmdb = tfidf_tmdb.fit_transform(tmdb_df['tags'])
knn_tmdb = NearestNeighbors(metric='cosine', algorithm='brute')
knn_tmdb.fit(X_tmdb)



print("Loading IMDb data...")
imdb_raw = pd.read_csv('movie_metadata.csv')


rating_src = imdb_raw[['movie_title', 'imdb_score']].copy()
rating_src['movie_title'] = (
    rating_src['movie_title']
    .astype(str)
    .str.replace(u'\xa0', ' ', regex=False)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+\(\d{4}\)', '', regex=True)
)
imdb_rating_dict = dict(zip(rating_src['movie_title'], rating_src['imdb_score']))


imdb = imdb_raw[['movie_title', 'genres', 'plot_keywords',
                  'director_name', 'actor_1_name',
                  'actor_2_name', 'actor_3_name']].copy()
imdb.dropna(inplace=True)

def pipe_to_list(obj):
    return [p.strip() for p in str(obj).split('|') if p.strip()]

def build_imdb_cast(row):
    L = []
    for col in ['actor_1_name', 'actor_2_name', 'actor_3_name']:
        if pd.notna(row[col]):
            L.append(str(row[col]))
    return L

imdb['genres_parsed'] = imdb['genres'].apply(pipe_to_list)
imdb['overview']      = imdb['plot_keywords'].apply(pipe_to_list)
imdb['keywords']      = [[] for _ in range(len(imdb))]
imdb['cast']          = imdb.apply(build_imdb_cast, axis=1)
imdb['crew']          = imdb['director_name'].apply(
                            lambda x: [str(x)] if pd.notna(x) else [])


imdb['genres_stored'] = imdb['genres_parsed'].apply(lambda x: [g.lower() for g in x])


imdb['genres_tags'] = imdb['genres_parsed'].apply(
                          lambda x: [g.replace(" ", "").lower() for g in x])
imdb['keywords']    = imdb['keywords'].apply(
                          lambda x: [i.replace(" ", "").lower() for i in x])
imdb['cast']        = imdb['cast'].apply(
                          lambda x: [i.replace(" ", "").lower() for i in x])
imdb['crew']        = imdb['crew'].apply(
                          lambda x: [i.replace(" ", "").lower() for i in x])

imdb['title'] = (imdb['movie_title']
                 .astype(str)
                 .str.replace(u'\xa0', ' ', regex=False)
                 .str.strip())

imdb['tags'] = (imdb['overview'] + imdb['genres_tags']
                + imdb['keywords'] + imdb['cast'] + imdb['crew'])

imdb_df = imdb[['title', 'tags', 'genres_stored']].copy()
imdb_df.rename(columns={'genres_stored': 'genres'}, inplace=True)

imdb_df['tags']        = imdb_df['tags'].apply(lambda x: " ".join(x).lower())
imdb_df['tags']        = imdb_df['tags'].apply(stem)
imdb_df['title_clean'] = imdb_df['title'].astype(str).str.lower().str.strip()
imdb_df = imdb_df.reset_index(drop=True)

print("Building IMDb TF-IDF + kNN model...")
tfidf_imdb = TfidfVectorizer(max_features=5000, stop_words='english')
X_imdb = tfidf_imdb.fit_transform(imdb_df['tags'])
knn_imdb = NearestNeighbors(metric='cosine', algorithm='brute')
knn_imdb.fit(X_imdb)



print("Loading Rotten Tomatoes data...")
rotten_raw = pd.read_csv('rotten_tomatoes_movies.csv')

rotten = rotten_raw[['movie_title', 'movie_info', 'genres',
                      'directors', 'actors']].copy()
rotten.dropna(inplace=True)

def rotten_genres_to_list(obj):
    return [p.strip() for p in str(obj).split(',') if p.strip()]

def rotten_top3_cast(obj):
    return [p.strip() for p in str(obj).split(',') if p.strip()][:3]

def rotten_directors_to_list(obj):
    return [p.strip() for p in str(obj).split(',') if p.strip()]

rotten['overview']      = rotten['movie_info'].apply(lambda x: str(x).split())
rotten['genres_parsed'] = rotten['genres'].apply(rotten_genres_to_list)
rotten['keywords']      = [[] for _ in range(len(rotten))]
rotten['cast']          = rotten['actors'].apply(rotten_top3_cast)
rotten['crew']          = rotten['directors'].apply(rotten_directors_to_list)


rotten['genres_stored'] = rotten['genres_parsed'].apply(
                              lambda x: [g.lower() for g in x])


rotten['genres_tags'] = rotten['genres_parsed'].apply(
                            lambda x: [g.replace(" ", "").lower() for g in x])
rotten['keywords']    = rotten['keywords'].apply(
                            lambda x: [i.replace(" ", "").lower() for i in x])
rotten['cast']        = rotten['cast'].apply(
                            lambda x: [i.replace(" ", "").lower() for i in x])
rotten['crew']        = rotten['crew'].apply(
                            lambda x: [i.replace(" ", "").lower() for i in x])

rotten['title'] = rotten['movie_title'].astype(str).str.strip()
rotten['tags']  = (rotten['overview'] + rotten['genres_tags']
                   + rotten['keywords'] + rotten['cast'] + rotten['crew'])

rotten_df = rotten[['title', 'tags', 'genres_stored']].copy()
rotten_df.rename(columns={'genres_stored': 'genres'}, inplace=True)

rotten_df['tags']        = rotten_df['tags'].apply(lambda x: " ".join(x).lower())
rotten_df['tags']        = rotten_df['tags'].apply(stem)
rotten_df['title_clean'] = rotten_df['title'].astype(str).str.lower().str.strip()
rotten_df = rotten_df.reset_index(drop=True)

print("Building Rotten Tomatoes TF-IDF + kNN model...")
tfidf_rotten = TfidfVectorizer(max_features=10000, stop_words='english')
X_rotten = tfidf_rotten.fit_transform(rotten_df['tags'])
knn_rotten = NearestNeighbors(metric='cosine', algorithm='brute')
knn_rotten.fit(X_rotten)




def recommend_from_tmdb(title, n=5):
    key = str(title).lower().strip()
    if key not in tmdb_df['title_clean'].values:
        print("Movie not found in TMDB dataset.")
        return []
    pos = tmdb_df[tmdb_df['title_clean'] == key].index[0]
    distances, indices = knn_tmdb.kneighbors(X_tmdb[pos], n_neighbors=n + 1)
    return tmdb_df.iloc[indices[0][1:]]['title'].tolist()

def recommend_from_imdb(title, n=5):
    key = str(title).lower().strip()
    if key not in imdb_df['title_clean'].values:
        print("Movie not found in IMDb dataset.")
        return []
    pos = imdb_df[imdb_df['title_clean'] == key].index[0]
    distances, indices = knn_imdb.kneighbors(X_imdb[pos], n_neighbors=n + 1)
    return imdb_df.iloc[indices[0][1:]]['title'].tolist()

def recommend_from_rotten(title, n=5):
    key = str(title).lower().strip()
    if key not in rotten_df['title_clean'].values:
        print("Movie not found in Rotten Tomatoes dataset.")
        return []
    pos = rotten_df[rotten_df['title_clean'] == key].index[0]
    distances, indices = knn_rotten.kneighbors(X_rotten[pos], n_neighbors=n + 1)
    return rotten_df.iloc[indices[0][1:]]['title'].tolist()

def recommend_all(title, n=10):
    key = str(title).lower().strip()
    rec_all = []
    if key in tmdb_df['title_clean'].values:
        rec_all += recommend_from_tmdb(title, n=n)
    if key in imdb_df['title_clean'].values:
        rec_all += recommend_from_imdb(title, n=n)
    if key in rotten_df['title_clean'].values:
        rec_all += recommend_from_rotten(title, n=n)
    if not rec_all:
        print("Movie not found in any dataset.")
        return []
    counts = Counter(rec_all)
    filtered = [(t, c) for t, c in counts.items() if t.lower().strip() != key]
    filtered_sorted = sorted(filtered, key=lambda x: (-x[1], x[0]))
    return [t for t, c in filtered_sorted][:n]



print("Saving all models and dataframes...")

pickle.dump(tmdb_df,       open('tmdb_df.pkl',        'wb'))
pickle.dump(tfidf_tmdb,    open('tfidf_tmdb.pkl',      'wb'))
pickle.dump(knn_tmdb,      open('knn_tmdb.pkl',        'wb'))

pickle.dump(imdb_df,       open('imdb_df.pkl',         'wb'))
pickle.dump(tfidf_imdb,    open('tfidf_imdb.pkl',      'wb'))
pickle.dump(knn_imdb,      open('knn_imdb.pkl',        'wb'))

pickle.dump(rotten_df,     open('rotten_df.pkl',       'wb'))
pickle.dump(tfidf_rotten,  open('tfidf_rotten.pkl',    'wb'))
pickle.dump(knn_rotten,    open('knn_rotten.pkl',      'wb'))

pickle.dump(imdb_rating_dict, open('imdb_rating_dict.pkl', 'wb'))

print("✅ All done! Pickles saved.")



if __name__ == "__main__":
    test_movie = "Avatar"
    print(f"\nTMDB recs:   {recommend_from_tmdb(test_movie, n=5)}")
    print(f"IMDb recs:   {recommend_from_imdb(test_movie, n=5)}")
    print(f"Rotten recs: {recommend_from_rotten(test_movie, n=5)}")
    print(f"Ensemble:    {recommend_all(test_movie, n=5)}")

Loading TMDB data...
Building TMDB TF-IDF + kNN model...
Loading IMDb data...
Building IMDb TF-IDF + kNN model...
Loading Rotten Tomatoes data...
Building Rotten Tomatoes TF-IDF + kNN model...
Saving all models and dataframes...
✅ All done! Pickles saved.

TMDB recs:   ['Aliens', 'Falcon Rising', 'Battle: Los Angeles', 'Aliens vs Predator: Requiem', 'Apollo 18']
IMDb recs:   ['Aliens', 'Mystery Men', 'The Ridges', 'The New World', 'Terminator 2: Judgment Day']
Rotten recs: ['The Blue Tooth Virgin', 'Nas: Time Is Illmatic', 'Graduation', 'Soul on a String', 'Wonderful Town']
Ensemble:    ['Aliens', 'Aliens vs Predator: Requiem', 'Apollo 18', 'Battle: Los Angeles', 'Falcon Rising']
